# Part 1 — Document Extraction & Prompt Engineering

See `README.md` for the full parsing justification, methodology, assumptions, and known limitations.
This notebook runs the extraction end-to-end and verifies every field against manually cross-checked source figures.

In [1]:
from extract import run_extraction
from llm_config import HAIKU_MODEL

result = run_extraction(model=HAIKU_MODEL, max_tokens=1024)
print(result.model_dump_json(indent=2))

2026-09-12 21:56:00,960 INFO run_extraction[claude-haiku-4-5-20251001]: start


2026-09-12 21:56:05,556 INFO run_extraction[claude-haiku-4-5-20251001]: done in 4.60s


{
  "corp_income_tax_2024_billion": 28.03,
  "corp_income_tax_yoy_pct": -1.2,
  "total_topups_2024_billion": 20.352,
  "operating_revenue_taxes": [
    "Corporate Income Tax",
    "Personal Income Tax",
    "Withholding Tax",
    "Statutory Boards' Contributions",
    "Assets Taxes",
    "Customs, Excise and Carbon Taxes",
    "Goods and Services Tax",
    "Motor Vehicle Taxes",
    "Vehicle Quota Premiums",
    "Betting Taxes",
    "Stamp Duty",
    "Other Taxes"
  ],
  "latest_actual_fiscal_position_billion": -3.57
}


## Verification against ground truth

Ground-truth values below were confirmed by manually reading the source PDF's tables directly
(Table 2.1 p.16 for CIT/YoY, Table 2.4 p.20 for top-ups, Table 1.1 p.8 for fiscal position,
narrative on pp.5-6 for the tax list) — not derived from the LLM's own output.

Note: the assignment brief cites page 5 for the CIT/YoY fields, but page 5 is FY2023 data.
The correct FY2024 figures are on page 16 — see `README.md` for details.

In [2]:
ground_truth = {
    "corp_income_tax_2024_billion": 28.03,
    "corp_income_tax_yoy_pct": -1.2,
    "total_topups_2024_billion": 20.352,
    "operating_revenue_taxes": [
        "Corporate Income Tax", "Personal Income Tax", "Withholding Tax",
        "Statutory Boards' Contributions", "Assets Taxes",
        "Customs, Excise and Carbon Taxes", "Goods and Services Tax",
        "Motor Vehicle Taxes", "Vehicle Quota Premiums", "Betting Taxes",
        "Stamp Duty", "Other Taxes",
    ],
    "latest_actual_fiscal_position_billion": -3.57,
}

extracted = result.model_dump()

print(f"{'Field':<42}{'Extracted':<20}{'Ground truth':<20}{'Match'}")
for field, gt_value in ground_truth.items():
    ex_value = extracted[field]
    if isinstance(gt_value, list):
        match = set(ex_value) == set(gt_value)
    else:
        match = abs(ex_value - gt_value) < 0.01
    print(f"{field:<42}{str(ex_value):<20}{str(gt_value):<20}{'PASS' if match else 'FAIL'}")

Field                                     Extracted           Ground truth        Match
corp_income_tax_2024_billion              28.03               28.03               PASS
corp_income_tax_yoy_pct                   -1.2                -1.2                PASS
total_topups_2024_billion                 20.352              20.352              PASS
operating_revenue_taxes                   ['Corporate Income Tax', 'Personal Income Tax', 'Withholding Tax', "Statutory Boards' Contributions", 'Assets Taxes', 'Customs, Excise and Carbon Taxes', 'Goods and Services Tax', 'Motor Vehicle Taxes', 'Vehicle Quota Premiums', 'Betting Taxes', 'Stamp Duty', 'Other Taxes']['Corporate Income Tax', 'Personal Income Tax', 'Withholding Tax', "Statutory Boards' Contributions", 'Assets Taxes', 'Customs, Excise and Carbon Taxes', 'Goods and Services Tax', 'Motor Vehicle Taxes', 'Vehicle Quota Premiums', 'Betting Taxes', 'Stamp Duty', 'Other Taxes']PASS
latest_actual_fiscal_position_billion     -3.57         